# 01 · Month-end close checks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammedelsayed741996-blip/berlin-data-portfolio/blob/main/agritech-corporate-finance/month-end-close-forecast/notebooks/01_month_end_close.ipynb)

Feldkern Group is a made-up seed breeding company with three entities (Germany, France, Austria) and a fiscal year from July to June. This notebook closes September 2026, period 3 of FY 2026/27, from a ledger extract shaped like an SAP line item table: company code, document, account, cost center, WBS element, partner company, vendor and amount.

Twelve checks run on the extract. Before the close, eleven errors were planted in September on purpose: a document that does not balance, a blocked cost center, a duplicate invoice, a missing depreciation run, a missing rent posting, an intercompany charge the partner never booked, and so on. The checks have to find all of them, and the earlier months have to come out clean.

In [1]:
import os, sys, subprocess, warnings
from pathlib import Path

REPO = "https://github.com/mohammedelsayed741996-blip/berlin-data-portfolio.git"
PROJECT = "agritech-corporate-finance/month-end-close-forecast"

if "google.colab" in sys.modules:
    if not Path("/content/berlin-data-portfolio").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/berlin-data-portfolio"], check=True)
    os.chdir(f"/content/berlin-data-portfolio/{PROJECT}")
else:
    here = Path.cwd()
    while not (here / "src" / "close_forecast").exists() and here != here.parent:
        here = here.parent
    os.chdir(here)

sys.path.insert(0, str(Path.cwd() / "src"))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
from IPython.display import Image, display
print("Project folder:", Path.cwd().name)

Project folder: month-end-close-forecast


In [2]:
from close_forecast import config as C, pipeline, close_checks, variance, forecast, reporting, db

tables = pipeline.load_tables()          # reads data/, or generates it if missing
run = pipeline.run(tables)
gl = tables["gl_items"]
print(f"{len(gl):,} ledger lines from {gl['month'].min()} to {gl['month'].max()}, closing {C.CLOSE_PERIOD}")

31,781 ledger lines from 2024-07 to 2026-09, closing 2026-09


## The checklist

FAIL means the close cannot be signed off. WARN means someone has to look at it and decide.

In [3]:
run.checklist[["check_id", "check", "owner", "status", "exceptions", "amount_eur"]]

,check_id,check,owner,status,exceptions,amount_eur
0,C01,Every document balances (debit = credit),General ledger,FAIL,1,"1,250.00"
1,C02,No postings on blocked cost centers,Controlling,FAIL,1,"8,900.00"
2,C03,Cost center belongs to the posting company,Controlling,FAIL,1,"12,400.00"
3,C04,Expense lines carry a cost center,Controlling,FAIL,1,"5,600.00"
4,C05,No duplicate vendor invoices,Accounts payable,FAIL,2,"42,350.00"
5,C06,Entries made by working day 3 (2026-10-05),General ledger,WARN,6,"37,026.01"
6,C07,"Recurring postings complete (payroll, depreciation, rent, IC)",General ledger,FAIL,3,"166,776.21"
7,C08,Intercompany balances agree by partner,Group accounting,FAIL,1,"19,331.50"
8,C09,GR/IR items older than 60 days,Procurement / AP,WARN,4,"92,986.73"
9,C10,Suspense account cleared to zero,General ledger,WARN,1,"7,800.00"


A month that nobody tampered with passes every check. That matters as much as finding the errors: a check that fires every month gets ignored.

In [4]:
close_checks.checklist(close_checks.run_close_checks(tables, "2025-12"))[["check_id", "status", "exceptions"]].T

,0,1,2,3,4,5,6,7,8,9,10,11
check_id,C01,C02,C03,C04,C05,C06,C07,C08,C09,C10,C11,C12
status,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS
exceptions,0,0,0,0,0,0,0,0,0,0,0,0


## What the checks found

The document that does not balance came in through an upload from a feeder system. SAP would reject it at entry, but uploads and interfaces are where this happens in practice.

In [5]:
checks = {c.check_id: c for c in run.checks}
for cid in ["C01", "C02", "C03", "C04", "C05"]:
    print(cid, checks[cid].name)
    display(checks[cid].exceptions)

C01 Every document balances (debit = credit)
C02 No postings on blocked cost centers
C03 Cost center belongs to the posting company
C04 Expense lines carry a cost center
C05 No duplicate vendor invoices


,company_code,doc_no,doc_type,text,difference_eur
450,DE10,1000009422,SA,Upload from consolidation feeder: audit fee accrual,"1,250.00"


,company_code,doc_no,gl_account,cost_center,amount_eur,text
31747,DE10,1000009423,6500,4199,"8,900.00",Reporting tool licence


,company_code,doc_no,gl_account,cost_center,cost_center_entity,amount_eur,text
31749,DE10,1000009424,6400,2300,FR10,"12,400.00",Trade fair travel Paris


,company_code,doc_no,gl_account,amount_eur,vendor,text
31751,FR10,3000003323,6500,"5,600.00",V0301,Legal advice


,company_code,doc_no,vendor,reference,posting_date,amount_eur,text
31754,DE10,1000009425,V0147,RE-2026-08817,2026-09-16,"42,350.00",Cloud hosting Q3
31756,DE10,1000009426,V0147,RE-2026-08817,2026-09-27,"42,350.00",Cloud hosting Q3


## Missing recurring postings

The close keeps a register of what must be posted every month: payroll and the depreciation run for each company, rent for each cost center with a lease, and the intercompany charges. In September three of them are missing. The amount shown is last month's, as an estimate of what is missing.

In [6]:
checks["C07"].exceptions

,company_code,posting,gl_account,cost_center,last_month_eur
0,AT10,depreciation run,6200,all,"111,933.59"
1,FR10,rent,6600,2300,"35,511.12"
2,AT10,IC IT recharge,6950,all,"19,331.50"


The intercompany check finds the same gap from the other side. DE10 billed Austria for IT services, Austria did not book the invoice, and the receivable and payable now differ by exactly that amount.

In [7]:
close_checks.intercompany_matrix(tables, C.CLOSE_PERIOD)

,pair,DE10_receivable_eur,partner_payable_eur,balance_difference_eur,month_income_eur,month_charge_eur,month_difference_eur
0,DE10-AT10,"202,980.75","183,649.25","19,331.50","67,660.25","48,328.75","19,331.50"
1,DE10-FR10,"310,774.89","310,774.89",0.00,"103,591.63","103,591.63",0.00


## Cut-off, GR/IR and suspense

Six invoices were entered on 7 October, after the cut-off on working day 3. Four goods receipts from May and June still have no invoice, which usually means either an invoice is stuck somewhere or the receipt was booked twice.

In [8]:
display(checks["C06"].exceptions)
display(close_checks.grir_open_items(tables, C.CLOSE_PERIOD).head(8))
display(checks["C10"].exceptions)

,company_code,doc_no,doc_type,entry_date,text,amount_eur
98,AT10,2000002741,KR,2026-10-07,Late invoice entered after cut-off,"6,537.65"
455,DE10,1000009427,KR,2026-10-07,Late invoice entered after cut-off,"6,053.39"
456,DE10,1000009428,KR,2026-10-07,Late invoice entered after cut-off,"4,769.16"
457,DE10,1000009429,KR,2026-10-07,Late invoice entered after cut-off,"4,967.70"
458,DE10,1000009430,KR,2026-10-07,Late invoice entered after cut-off,"10,686.68"
593,FR10,3000003325,KR,2026-10-07,Late invoice entered after cut-off,"4,011.43"


,company_code,po_number,open_eur,first_receipt,age_days
953,DE10,4500001173,"24,921.42",2026-05-05,148
954,DE10,4500001174,"25,954.00",2026-05-10,143
1175,FR10,4500001175,"22,034.53",2026-06-15,107
955,DE10,4500001176,"20,076.78",2026-06-20,102
932,DE10,4500001133,"175,532.77",2026-09-12,18
937,DE10,4500001138,"115,479.00",2026-09-18,12
1166,FR10,4500001164,"72,365.75",2026-09-23,7
931,DE10,4500001132,"204,631.65",2026-09-30,0


,company_code,balance_eur
0,FR10,"7,800.00"


## Proposed corrections

The checks do not post anything. They propose entries, and an accountant approves them. Reclasses net to zero, so only the missing postings and the reversed duplicate change profit. The forecast in notebook 03 is built on actuals after these corrections, so known errors do not leak into the full-year outlook.

In [9]:
adj = run.adjustments
display(adj[["check_id", "company_code", "cost_center", "gl_account", "amount_eur", "description"]])
print(f"Net P&L effect: EUR {adj['amount_eur'].sum():,.2f}")

Net P&L effect: EUR 124,426.21


,check_id,company_code,cost_center,gl_account,amount_eur,description
0,C07,AT10,1200,6200,"97,813.19",Post missing depreciation run (estimate: last month)
1,C07,AT10,3200,6200,"14,120.40",Post missing depreciation run (estimate: last month)
2,C07,FR10,2300,6600,"35,511.12",Post missing rent (estimate: last month)
3,C07,AT10,4200,6950,"19,331.50",Post missing IC IT recharge (estimate: last month)
4,C05,DE10,4120,6500,"-42,350.00",Reverse duplicate invoice RE-2026-08817
5,C02,DE10,4199,6500,"-8,900.00",Move off blocked cost center
6,C02,DE10,4100,6500,"8,900.00",Rebook on successor cost center 4100
7,C03,DE10,2300,6400,"-12,400.00",Remove FR10 cost center from DE10 posting
8,C03,DE10,2120,6400,"12,400.00",Rebook on own selling cost center
9,C04,FR10,4300,6500,"5,600.00",Assign cost center (pending confirmation from requester)


## The same checks in SQL

The `sql/` folder has the core checks as plain SQL, run here on SQLite. The tests make sure SQL and Python give the same answer.

In [10]:
conn = db.connect(tables)
display(db.run_sql(conn, "03_intercompany_reconciliation.sql"))
display(db.run_sql(conn, "04_grir_aging.sql").groupby("bucket")["open_eur"].agg(["count", "sum"]))

,pair,de10_receivable_eur,partner_payable_eur,difference_eur
0,DE10-AT10,"202,980.75","183,649.25","19,331.50"
1,DE10-FR10,"310,774.89","310,774.89",0.00


,count,sum
bucket,,
0-30,4,"568,009.17"
90+,4,"92,986.73"


In [11]:
tb = close_checks.trial_balance(tables)
tb.groupby("company_code")["amount_eur"].sum().round(2)

company_code
AT10       0.00
DE10   1,250.00
FR10      -0.00
Name: amount_eur, dtype: float64

The trial balance nets to zero in Austria and France. Germany is off by EUR 1,250, the unbalanced upload from check C01.